# Milestone 6D: Real-Time Structured Streaming & Anomaly Detection

Demonstrates real-time transaction ingestion, watermarking, tumbling window aggregations, and near-real-time AML alerts using **Spark Structured Streaming**.

### Streaming Architecture:
1. **File Stream Source**: Monitors `data/stream/incoming/` micro-batches.
2. **Event-Time Watermark**: 10-minute tolerance for late-arriving events.
3. **5-Minute Tumbling Windows**: Computes rolling transaction count, unique senders, and volume.
4. **Anomaly Rule Filter**: Flags rapid fan-in signatures.
5. **Sinks**: Interactive console output and persistent Parquet sink.


In [ ]:
import os
import sys
import time
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../src'))

from src.streaming_pipeline import create_spark_session, StreamingAnomalyDetector
from src.stream_simulator import simulate_stream

spark = create_spark_session('Databricks-M6D-Streaming')
detector = StreamingAnomalyDetector(spark)
print('Incoming directory:', detector.incoming_dir)
print('Output directory:  ', detector.output_dir)


## 1. Emit Simulated Micro-Batches
Emit 3 sample micro-batches of transactions into the streaming incoming directory.


In [ ]:
source_csv = '../data/processed/processed_df.csv'
if not os.path.exists(source_csv):
    source_csv = '../data/raw/HI-Small_Trans.csv'

simulate_stream(source_csv, detector.incoming_dir, batch_size=200, delay_seconds=0.5, max_batches=3)
print('Sample micro-batches successfully generated.')


## 2. Start Structured Streaming Engine
Process micro-batches with tumbling 5-minute event windows and evaluate anomaly thresholds.


In [ ]:
queries = detector.start_pipeline(
    enable_console=True,
    enable_parquet=True,
    trigger_interval='2 seconds',
    timeout_seconds=10,
)
print('Streaming execution completed.')
spark.stop()
